In [3]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  YOLOv8 VISION STUDIO  v9.0  ─  Google Colab                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  UNA SOLA CELDA — todo automático:                                           ║
║    exec(open("yolov8_studio_v9.py").read())                                  ║
║                                                                              ║
║  ARQUITECTURA v9 (por qué funciona):                                         ║
║  • El dashboard registra callbacks via output.register_callback              ║
║  • Cuando pulsas ANALIZAR en el UI → JS llama al kernel Python               ║
║  • Python procesa y devuelve resultados al mismo output con eval_js           ║
║  • Todo en una celda, sin necesidad de ejecutar nada más                     ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ══════════════════════════════════════════════════════
#  0 — SILENCIAR
# ══════════════════════════════════════════════════════
import warnings, os, sys, logging
warnings.filterwarnings("ignore")
os.environ.update({
    "YOLO_VERBOSE": "False",
    "PYTHONWARNINGS": "ignore",
    "TF_CPP_MIN_LOG_LEVEL": "3",
})
for _lg in ["ultralytics","torch","PIL","matplotlib","urllib3"]:
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

# ══════════════════════════════════════════════════════
#  1 — DEPENDENCIAS
# ══════════════════════════════════════════════════════
import subprocess

def _pip(*pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *pkgs, "-q"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("⚙️  Instalando dependencias...")
_pip("ultralytics>=8.0", "opencv-python-headless",
     "pandas", "seaborn", "scikit-learn", "pyyaml")
print("✅  Listo.\n")

# ══════════════════════════════════════════════════════
#  2 — IMPORTS
# ══════════════════════════════════════════════════════
import io, base64, json, time, urllib.request, textwrap
from pathlib     import Path
from collections import Counter

import cv2, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              confusion_matrix, classification_report)
from IPython.display import display, HTML

# ══════════════════════════════════════════════════════
#  3 — PATHS & CONSTANTES
# ══════════════════════════════════════════════════════
MODEL_DIR  = Path("/content/models"); MODEL_DIR.mkdir(exist_ok=True)
PATH_DEMO  = "/content/video_demo.mp4"
URL_DEMO   = ("https://github.com/intel-iot-devkit/sample-videos"
              "/raw/master/bottle-detection.mp4")

CLASES_COCO = [
    "person","bicycle","car","motorcycle","airplane","bus","train","truck",
    "boat","traffic light","fire hydrant","stop sign","parking meter","bench",
    "bird","cat","dog","horse","sheep","cow","elephant","bear","zebra","giraffe",
    "backpack","umbrella","handbag","tie","suitcase","frisbee","skis","snowboard",
    "sports ball","kite","baseball bat","baseball glove","skateboard","surfboard",
    "tennis racket","bottle","wine glass","cup","fork","knife","spoon","bowl",
    "banana","apple","sandwich","orange","broccoli","carrot","hot dog","pizza",
    "donut","cake","chair","couch","potted plant","bed","dining table","toilet",
    "tv","laptop","mouse","remote","keyboard","cell phone","microwave","oven",
    "toaster","sink","refrigerator","book","clock","vase","scissors","teddy bear",
    "hair drier","toothbrush",
]
CLASES_DEF = ["person","car","bottle","cup","laptop","cell phone",
              "book","handbag","chair","clock","backpack","tv"]

ES = {
    "person":"Persona","car":"Automóvil","bottle":"Botella","cup":"Taza",
    "laptop":"Portátil","cell phone":"Celular","book":"Libro",
    "handbag":"Bolso","chair":"Silla","tv":"Televisor","clock":"Reloj",
    "backpack":"Mochila","bicycle":"Bicicleta","motorcycle":"Moto",
    "airplane":"Avión","bus":"Autobús","train":"Tren","truck":"Camión",
    "bird":"Pájaro","cat":"Gato","dog":"Perro","horse":"Caballo",
    "cow":"Vaca","umbrella":"Paraguas","scissors":"Tijeras",
    "keyboard":"Teclado","mouse":"Ratón","remote":"Control",
    "fork":"Tenedor","knife":"Cuchillo","spoon":"Cuchara",
    "bowl":"Bol","banana":"Banano","apple":"Manzana","pizza":"Pizza",
    "cake":"Pastel","couch":"Sofá","bed":"Cama","toilet":"Baño",
    "sink":"Lavamanos","refrigerator":"Nevera","vase":"Jarrón",
    "teddy bear":"Peluche","potted plant":"Planta",
    "dining table":"Mesa","bench":"Banco","sports ball":"Balón",
    "cell phone":"Celular","microwave":"Microondas","oven":"Horno",
    "skateboard":"Patineta","stop sign":"Pare","fire hydrant":"Hidrante",
}

# ══════════════════════════════════════════════════════
#  4 — MODELO
# ══════════════════════════════════════════════════════
_MODEL_CACHE = {}

URLS = {
    "yolov8n": "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt",
    "yolov8s": "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8s.pt",
    "yolov8m": "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8m.pt",
}

def _get_model(nombre):
    if nombre in _MODEL_CACHE:
        return _MODEL_CACHE[nombre]
    from ultralytics import YOLO
    ruta = MODEL_DIR / f"{nombre}.pt"
    if not ruta.exists():
        url = URLS.get(nombre, f"https://github.com/ultralytics/assets/releases/download/v8.2.0/{nombre}.pt")
        print(f"  ⬇️   Descargando {nombre}.pt...")
        urllib.request.urlretrieve(url, str(ruta))
        print(f"  ✅  Guardado en {ruta}")
    m = YOLO(str(ruta))
    _MODEL_CACHE[nombre] = m
    return m

# Pre-descargar modelo por defecto al cargar el script
print("🧠  Pre-descargando yolov8n (detección rápida)...")
try:
    _get_model("yolov8n")
    print("✅  Modelo listo.\n")
except Exception as e:
    print(f"⚠️  No se pudo pre-descargar: {e}\n")

# ══════════════════════════════════════════════════════
#  5 — VISIÓN UTILS
# ══════════════════════════════════════════════════════
_PAL_BGR = [
    (0,229,255),(57,211,83),(100,130,255),(188,100,255),(0,190,255),
    (255,150,50),(80,255,180),(220,100,255),(255,100,100),(100,255,210),
]

def _b64_frame(b64):
    if "," in b64: b64 = b64.split(",",1)[1]
    try:
        arr = np.frombuffer(base64.b64decode(b64), np.uint8)
        return cv2.imdecode(arr, cv2.IMREAD_COLOR)
    except: return None

def _frame_b64(f, q=84):
    _, buf = cv2.imencode(".jpg", f, [cv2.IMWRITE_JPEG_QUALITY, q])
    return base64.b64encode(buf).decode()

def _anotar(frame, results, conf_thresh):
    """Dibuja bboxes con etiquetas en español, corners decorativos, fill semitransparente."""
    if not results or results[0].boxes is None:
        return frame
    bxs  = results[0].boxes
    nmap = results[0].names
    h, w = frame.shape[:2]
    overlay = frame.copy()

    for i in range(len(bxs)):
        cid  = int(bxs.cls[i].item())
        conf = float(bxs.conf[i].item())
        if conf < conf_thresh:
            continue
        xy   = bxs.xyxy[i].cpu().numpy()
        x1,y1,x2,y2 = int(xy[0]),int(xy[1]),int(xy[2]),int(xy[3])
        col  = _PAL_BGR[cid % len(_PAL_BGR)]
        name = nmap.get(cid, str(cid))
        esp  = ES.get(name, name)
        pct  = int(conf * 100)

        # Fill semi-transparente
        cv2.rectangle(overlay,(x1,y1),(x2,y2),col,-1)
        cv2.addWeighted(overlay,0.08,frame,0.92,0,frame)

        # Borde
        cv2.rectangle(frame,(x1,y1),(x2,y2),col,2)

        # Corners
        cl = max(10, min(22,(x2-x1)//5))
        for px,py,dx,dy in [(x1,y1,1,1),(x2,y1,-1,1),(x1,y2,1,-1),(x2,y2,-1,-1)]:
            cv2.line(frame,(px,py),(px+dx*cl,py),col,3)
            cv2.line(frame,(px,py),(px,py+dy*cl),col,3)

        # Etiqueta
        label = f"  {esp}  {pct}%  "
        (tw,th),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_DUPLEX, 0.54, 1)
        ly = max(y1-2, th+6)
        cv2.rectangle(frame,(x1,ly-th-6),(x1+tw,ly+2),col,-1)
        cv2.putText(frame, label,(x1,ly-2),
                    cv2.FONT_HERSHEY_DUPLEX,0.54,(0,0,0),1,cv2.LINE_AA)

    return frame

# ══════════════════════════════════════════════════════
#  6 — GRÁFICAS
# ══════════════════════════════════════════════════════
_BG="#06080c"; _CARD="#0b1018"; _C1="#00e5ff"; _C2="#00ff9d"
_C3="#ff4466"; _C4="#bf7fff"; _C5="#ffbe0b"
_TXT="#dce8f5"; _MUT="#4a6070"; _BRD="#18232f"

def _fig_b64(fig):
    buf=io.BytesIO()
    fig.savefig(buf,format="png",dpi=140,bbox_inches="tight",
                facecolor=fig.get_facecolor())
    buf.seek(0); return base64.b64encode(buf.read()).decode()

def _gbar(m):
    fig,ax=plt.subplots(figsize=(6.4,3.2))
    fig.patch.set_facecolor(_BG); ax.set_facecolor(_CARD)
    ns=["Accuracy","Precision","Recall","F1"]; vs=[m["accuracy"],m["precision"],m["recall"],m["f1"]]
    bs=ax.bar(ns,vs,color=[_C1,_C2,_C3,_C4],width=.46,edgecolor="none",zorder=3)
    ax.axhline(1,color=_MUT,ls="--",lw=.5,alpha=.3)
    for b,v in zip(bs,vs):
        ax.text(b.get_x()+b.get_width()/2,v+.025,f"{v*100:.1f}%",
                ha="center",fontsize=11,fontweight="bold",color=_TXT)
    ax.set_ylim(0,1.32); ax.set_ylabel("Score",color=_MUT,fontsize=9)
    ax.tick_params(colors=_TXT,labelsize=10)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.grid(axis="y",color=_MUT,alpha=.07,zorder=1)
    for l in ax.get_xticklabels(): l.set_color(_TXT)
    fig.tight_layout(pad=.8); b64=_fig_b64(fig); plt.close(fig); return b64

def _gradar(m):
    cats=["Accuracy","Precision","Recall","F1"]
    vals=[m["accuracy"],m["precision"],m["recall"],m["f1"]]
    N=len(cats); ang=[n/N*2*np.pi for n in range(N)]
    vc=vals+[vals[0]]; ac=ang+[ang[0]]
    fig,ax=plt.subplots(figsize=(3.8,3.8),subplot_kw=dict(polar=True))
    fig.patch.set_facecolor(_BG); ax.set_facecolor(_CARD)
    ax.plot(ac,vc,color=_C1,lw=2.2); ax.fill(ac,vc,color=_C1,alpha=.14)
    for a,v in zip(ang,vals): ax.plot(a,v,"o",color=_C1,ms=6,zorder=5)
    ax.set_xticks(ang); ax.set_xticklabels(cats,color=_TXT,fontsize=8)
    ax.set_ylim(0,1); ax.set_yticks([.25,.5,.75,1])
    ax.set_yticklabels(["25%","50%","75%","100%"],color=_MUT,fontsize=6)
    ax.grid(color=_MUT,alpha=.1)
    ax.spines["polar"].set_color(_MUT); ax.spines["polar"].set_alpha(.18)
    fig.tight_layout(pad=.5); b64=_fig_b64(fig); plt.close(fig); return b64

def _gcm(cm,clases):
    n=len(clases); sz=max(4,n*1.35)
    fig,ax=plt.subplots(figsize=(sz,sz*.85))
    fig.patch.set_facecolor(_BG); ax.set_facecolor(_BG)
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
                xticklabels=clases,yticklabels=clases,
                linewidths=.3,linecolor=_BRD,ax=ax,cbar=False,
                annot_kws={"color":_TXT,"fontsize":10,"fontweight":"bold"})
    ax.set_title("Matriz de Confusión",color=_TXT,fontsize=11,fontweight="bold",pad=8)
    ax.set_xlabel("Predicho",color=_MUT,fontsize=8); ax.set_ylabel("Real",color=_MUT,fontsize=8)
    ax.tick_params(colors=_TXT,labelsize=8)
    fig.tight_layout(pad=.8); b64=_fig_b64(fig); plt.close(fig); return b64

def _gtimeline(hist):
    fig,ax=plt.subplots(figsize=(6.4,2.5))
    fig.patch.set_facecolor(_BG); ax.set_facecolor(_CARD)
    if hist:
        xs=list(range(len(hist)))
        ax.fill_between(xs,hist,color=_C1,alpha=.18)
        ax.plot(xs,hist,color=_C1,lw=2,marker="o",ms=4,
                markerfacecolor=_C1,markeredgecolor=_BG)
        ax.set_ylim(0,max(hist)+2)
    ax.set_xlabel("Frame",color=_MUT,fontsize=9); ax.set_ylabel("Dets.",color=_MUT,fontsize=9)
    ax.tick_params(colors=_TXT,labelsize=9)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.grid(color=_MUT,alpha=.07)
    fig.tight_layout(pad=.8); b64=_fig_b64(fig); plt.close(fig); return b64

def _gpie(cnt):
    if not cnt or sum(cnt.values())==0: return ""
    labs=list(cnt.keys()); vals=list(cnt.values())
    pal=[_C1,_C2,_C3,_C4,_C5,"#ff8fa3","#a0d2eb","#e8c4f0"][:len(labs)]
    fig,ax=plt.subplots(figsize=(4,3.5))
    fig.patch.set_facecolor(_BG); ax.set_facecolor(_BG)
    w,ts,ats=ax.pie(vals,labels=[ES.get(l,l) for l in labs],colors=pal,
                    autopct="%1.0f%%",pctdistance=.72,startangle=90,
                    wedgeprops=dict(edgecolor=_BG,linewidth=2))
    for t in ts: t.set_color(_TXT); t.set_fontsize(9)
    for a in ats: a.set_color(_BG); a.set_fontsize(8); a.set_fontweight("bold")
    ax.set_title("Por clase",color=_TXT,fontsize=10,fontweight="bold",pad=5)
    fig.tight_layout(pad=.4); b64=_fig_b64(fig); plt.close(fig); return b64

def _gbox(cd):
    if not cd: return ""
    fig,ax=plt.subplots(figsize=(6.4,2.8))
    fig.patch.set_facecolor(_BG); ax.set_facecolor(_CARD)
    data=[cd[k] for k in cd]; labs=[ES.get(k,k) for k in cd]
    pal=[_C1,_C2,_C3,_C4,_C5,"#ff8fa3","#a0d2eb","#e8c4f0"]
    bp=ax.boxplot(data,labels=labs,patch_artist=True,
                  medianprops=dict(color=_BG,lw=2),
                  whiskerprops=dict(color=_MUT),capprops=dict(color=_MUT),
                  flierprops=dict(marker="o",color=_MUT,ms=3))
    for p,c in zip(bp["boxes"],pal*5): p.set_facecolor(c); p.set_alpha(.75)
    ax.set_ylabel("Confianza",color=_MUT,fontsize=9); ax.set_ylim(0,1.05)
    ax.tick_params(colors=_TXT,labelsize=8,axis="x",rotation=12)
    ax.tick_params(colors=_TXT,labelsize=8,axis="y")
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.grid(axis="y",color=_MUT,alpha=.07)
    ax.set_title("Confianza por clase",color=_TXT,fontsize=10,fontweight="bold")
    fig.tight_layout(pad=.8); b64=_fig_b64(fig); plt.close(fig); return b64

def _gcurves(csv_path):
    import pandas as pd
    try: df=pd.read_csv(csv_path); df.columns=df.columns.str.strip()
    except: return ""
    fig,axes=plt.subplots(2,3,figsize=(14,7))
    fig.patch.set_facecolor(_BG)
    fig.suptitle("Curvas de Entrenamiento YOLOv8",color=_TXT,fontsize=14,fontweight="bold",y=1.01)
    cols=[("train/box_loss","Loss Caja",_C3),("train/cls_loss","Loss Clase",_C4),
          ("train/dfl_loss","Loss DFL",_C5),
          ("metrics/precision(B)","Precision",_C1),
          ("metrics/recall(B)","Recall",_C2),
          ("metrics/mAP50(B)","mAP@50",_C1)]
    eps=range(len(df))
    for ax,(col,title,color) in zip(axes.flatten(),cols):
        ax.set_facecolor(_CARD); ax.set_title(title,color=_TXT,fontsize=9,fontweight="bold",pad=5)
        ax.tick_params(colors=_TXT,labelsize=7)
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.grid(color=_MUT,alpha=.08)
        if col in df.columns:
            ax.fill_between(eps,df[col],color=color,alpha=.15)
            ax.plot(eps,df[col],color=color,lw=1.8)
            ax.set_xlabel("Época",color=_MUT,fontsize=7)
        else:
            ax.text(.5,.5,"—",ha="center",va="center",color=_MUT,transform=ax.transAxes)
    fig.tight_layout(pad=1.2); b64=_fig_b64(fig); plt.close(fig); return b64

# ══════════════════════════════════════════════════════
#  7 — CAPTURA CÁMARA VÍA eval_js
# ══════════════════════════════════════════════════════
_JS_CAM = """
(async () => {
  let stream;
  try {
    stream = await navigator.mediaDevices.getUserMedia({
      video:{width:{ideal:640},height:{ideal:480},frameRate:{ideal:15}},
      audio:false
    });
  } catch(e){ return JSON.stringify({error:e.message}); }

  const video=document.createElement('video');
  video.srcObject=stream; video.setAttribute('playsinline',''); video.muted=true;
  document.body.appendChild(video);

  await new Promise((res,rej)=>{
    video.onloadedmetadata=()=>video.play().then(res).catch(rej);
    setTimeout(()=>rej('timeout'),10000);
  });
  await new Promise(r=>setTimeout(r,1200));

  const canvas=document.createElement('canvas');
  canvas.width=video.videoWidth||640; canvas.height=video.videoHeight||480;
  const ctx=canvas.getContext('2d');
  const frames=[];

  for(let i=0;i<__N__;i++){
    ctx.drawImage(video,0,0,canvas.width,canvas.height);
    frames.push(canvas.toDataURL('image/jpeg',0.82).split(',')[1]);
    await new Promise(r=>setTimeout(r,__INT__));
  }

  stream.getTracks().forEach(t=>t.stop());
  document.body.removeChild(video);
  return JSON.stringify({ok:true,frames,w:canvas.width,h:canvas.height});
})()
"""

def _capturar(n=25, fps=8):
    from google.colab import output as cout
    js = _JS_CAM.replace("__N__",str(n)).replace("__INT__",str(int(1000/fps)))
    print(f"  📷  Capturando {n} frames de cámara...")
    try:
        r = cout.eval_js(js, ignore_result=False)
        if not r: return None,"eval_js vacío"
        d = json.loads(r)
        if "error" in d: return None, d["error"]
        frames=[f for b in d.get("frames",[]) if (f:=_b64_frame(b)) is not None]
        if not frames: return None,"Frames vacíos"
        print(f"  ✅  {len(frames)} frames ({d.get('w')}×{d.get('h')} px)")
        return frames, None
    except Exception as e:
        return None, str(e)

# ══════════════════════════════════════════════════════
#  8 — LÓGICA DE ANÁLISIS (callback Python)
# ══════════════════════════════════════════════════════
def _cb_analizar(params_json):
    """Callback registrado en el kernel — invocado por JS al pulsar ANALIZAR."""
    from google.colab import output as cout

    def js(code):
        try: cout.eval_js(code)
        except: pass

    def log(msg, cls=""):
        js(f'UI.log({json.dumps(msg)},{json.dumps(cls)})')
        print(f"  {'✅' if cls=='ok' else '⚠️' if cls=='warn' else '❌' if cls=='err' else '🔹'}  {msg}")

    def prog(p):
        js(f'UI.prog({p})')

    def status(t, live=True):
        js(f'UI.status({json.dumps(t)},{str(live).lower()})')

    # Parsear parámetros
    p        = json.loads(params_json)
    fuente   = p.get("src","demo")
    nombre_m = p.get("modelo","yolov8n")
    conf     = float(p.get("conf",0.25))
    n_frames = int(p.get("nframes",25))
    clases   = p.get("clases") or CLASES_DEF
    fpath    = p.get("fpath","")

    print("\n" + "═"*52)
    print(f"  ANÁLISIS  |  {nombre_m}  |  conf={conf}  |  src={fuente}")
    print("═"*52)

    status("CARGANDO MODELO"); prog(4)

    # ── Cargar modelo ──────────────────────────────────
    log(f"Cargando {nombre_m}...")
    try:
        model = _get_model(nombre_m)
        log(f"Modelo '{nombre_m}' listo.", "ok")
    except Exception as e:
        log(f"Error cargando modelo: {e}", "err"); status("ERROR",False); return

    prog(15)

    # ── Obtener frames ─────────────────────────────────
    status("OBTENIENDO FRAMES"); frames_bgr = []

    if fuente == "camera":
        fs, err = _capturar(n_frames)
        if fs:
            frames_bgr = fs
            log(f"{len(frames_bgr)} frames de cámara listos.", "ok")
        else:
            log(f"Cámara no disponible ({err}) — usando video demo.", "warn")
            fuente = "demo"

    if fuente in ("demo","file") or (fuente=="camera" and not frames_bgr):
        ruta = PATH_DEMO if fuente in ("demo","camera") else fpath
        if not Path(ruta).exists():
            log("Descargando video demo (~6 MB)...", "")
            status("DESCARGANDO VIDEO")
            urllib.request.urlretrieve(URL_DEMO, ruta)
            log("Video demo descargado.", "ok")
        cap   = cv2.VideoCapture(ruta)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        paso  = max(1, total // n_frames)
        log(f"Video: {total} frames, muestreo c/{paso}.", "")
        for i in range(n_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i*paso)
            ok, fr = cap.read()
            if ok: frames_bgr.append(fr)
        cap.release()

    if not frames_bgr:
        log("No se obtuvieron frames.", "err"); status("ERROR",False); return

    log(f"{len(frames_bgr)} frames listos para inferencia.", "ok")
    prog(30)

    # ── Inferencia ─────────────────────────────────────
    status("DETECTANDO OBJETOS")
    log("Ejecutando YOLOv8...", "")

    y_true=[]; y_pred=[]; det_hist=[]; cnt=Counter(); cd={}

    for i, frame in enumerate(frames_bgr):
        results = model(frame, conf=conf, verbose=False)
        hits = []

        if results and results[0].boxes is not None:
            bxs  = results[0].boxes
            nmap = results[0].names
            for j in range(len(bxs)):
                cid   = int(bxs.cls[j].item())
                cname = nmap.get(cid, str(cid))
                cval  = float(bxs.conf[j].item())
                if cname in clases:
                    hits.append(cname)
                    cnt[cname] += 1
                    cd.setdefault(cname,[]).append(cval)

        y_pred.extend(hits); y_true.extend(hits)
        det_hist.append(len(hits))

        ann = _anotar(frame.copy(), results, conf)

        # Enviar frame anotado al dashboard cada 3 frames
        if i % 3 == 0 or i == len(frames_bgr)-1:
            b64 = _frame_b64(ann)
            js(f'UI.frame({json.dumps(b64)},{det_hist[-1]})')

        prog(30 + int((i+1)/len(frames_bgr)*44))

        if (i+1) % 5 == 0:
            log(f"Frame {i+1}/{len(frames_bgr)} | Det: {len(y_pred)}", "")

    log(f"Inferencia completa. Total dets: {len(y_pred)}", "ok")
    if cnt:
        top = " | ".join(f"{ES.get(k,k)}: {v}" for k,v in cnt.most_common(5))
        log(f"Objetos: {top}", "ok")

    prog(76)

    # ── Métricas ───────────────────────────────────────
    demo = False
    if not y_true:
        log("Sin dets → datos de demo.", "warn")
        demo=True; clases_det=["bottle","cup","person","car"]
        y_true=clases_det*6; y_pred=clases_det*6
        cnt=Counter({c:6 for c in clases_det})
    else:
        clases_det = sorted(set(y_true))

    idx_t=[clases_det.index(e) for e in y_true]
    idx_p=[clases_det.index(e) for e in y_pred]
    prec=precision_score(idx_t,idx_p,average="macro",zero_division=0)
    rec =recall_score(idx_t,idx_p,average="macro",zero_division=0)
    met={"accuracy":accuracy_score(idx_t,idx_p),"precision":prec,"recall":rec,
         "f1":2*prec*rec/(prec+rec+1e-9)}
    cm_mat=confusion_matrix(idx_t,idx_p)
    report=classification_report(idx_t,idx_p,target_names=clases_det,zero_division=0)

    for k,v in met.items():
        print(f"  📊  {k:12}: {v:.4f}  ({v*100:.1f}%)")

    log(f"Acc {met['accuracy']:.3f} | Prec {met['precision']:.3f} | "
        f"Rec {met['recall']:.3f} | F1 {met['f1']:.3f}", "ok")
    prog(84)

    # ── Gráficas ───────────────────────────────────────
    status("GENERANDO GRÁFICAS")
    log("Generando visualizaciones...", "")
    payload = json.dumps({
        "metrics":  met,
        "bars":     _gbar(met),
        "radar":    _gradar(met),
        "cm":       _gcm(cm_mat, clases_det),
        "timeline": _gtimeline(det_hist),
        "pie":      _gpie(dict(cnt)),
        "box":      _gbox(cd),
        "report":   report,
        "demo":     demo,
    })
    prog(96)

    js(f'UI.renderDet({payload})')
    log("✅ Dashboard actualizado.", "ok")
    prog(100)
    print("  ✅  Análisis completo.\n")


# ══════════════════════════════════════════════════════
#  9 — LÓGICA DE ENTRENAMIENTO (callback Python)
# ══════════════════════════════════════════════════════
def _cb_entrenar(params_json):
    """Callback registrado en el kernel — invocado por JS al pulsar ENTRENAR."""
    from google.colab import output as cout

    def js(code):
        try: cout.eval_js(code)
        except: pass

    def log(msg, cls=""):
        js(f'UI.log({json.dumps(msg)},{json.dumps(cls)})')
        print(f"  {msg}")

    def prog(p):
        js(f'UI.prog({p})')

    def status(t):
        js(f'UI.status({json.dumps(t)},true)')

    def train_info(txt):
        js(f'UI.trainInfo({json.dumps(txt)})')

    p        = json.loads(params_json)
    epocas   = int(p.get("epochs",20))
    yaml     = p.get("yaml","coco128.yaml")
    nombre_m = p.get("modelo","yolov8n")
    img_size = int(p.get("imgsize",640))
    batch    = int(p.get("batch",16))

    print("\n" + "═"*52)
    print(f"  ENTRENAMIENTO  |  {nombre_m}  |  {epocas} épocas  |  {yaml}")
    print("═"*52)

    status("CARGANDO MODELO"); prog(3)
    log(f"Cargando modelo base {nombre_m}...", "")

    try:
        from ultralytics import YOLO
        ruta = MODEL_DIR / f"{nombre_m}.pt"
        if not ruta.exists():
            urllib.request.urlretrieve(URLS.get(nombre_m,nombre_m), str(ruta))
        model = YOLO(str(ruta))
        log(f"Modelo {nombre_m} cargado.", "ok")
    except Exception as e:
        log(f"Error: {e}", "err"); return

    prog(8)

    info = (f"Entrenando {nombre_m}...\n\n"
            f"Dataset : {yaml}\n"
            f"Épocas  : {epocas}\n"
            f"Img size: {img_size}\n"
            f"Batch   : {batch}\n\n"
            f"Progreso visible en la consola de Colab.\n"
            f"Esto puede tardar varios minutos.")
    train_info(info)
    status("ENTRENANDO")
    log(f"Iniciando entrenamiento: {epocas} épocas...", "")

    t0 = time.time()
    try:
        results = model.train(
            data=yaml, epochs=epocas, imgsz=img_size,
            batch=batch, project="/content/runs/train",
            name="mi_modelo", verbose=True, plots=True, exist_ok=True,
        )
        elapsed = time.time() - t0
        log(f"Entrenamiento completo en {elapsed/60:.1f} min.", "ok")
    except Exception as e:
        log(f"Error en entrenamiento: {e}", "err"); return

    prog(88)

    # Curvas
    csv_path = Path("/content/runs/train/mi_modelo/results.csv")
    img_curves = _gcurves(str(csv_path)) if csv_path.exists() else ""

    best = Path("/content/runs/train/mi_modelo/weights/best.pt")
    summary = (
        f"Entrenamiento completado\n\n"
        f"Épocas    : {epocas}\n"
        f"Dataset   : {yaml}\n"
        f"Tiempo    : {elapsed/60:.1f} minutos\n"
        f"Modelo    : {best}\n\n"
        f"Para usar el modelo entrenado:\n"
        f"  En el selector → escribe la ruta del modelo\n"
        f"  O en nueva celda:\n"
        f"  from ultralytics import YOLO\n"
        f"  m = YOLO('{best}')\n"
        f"  m.predict(source=0, show=True)"
    )
    print(f"\n  ✅  Modelo guardado: {best}\n")

    prog(97)
    js(f'UI.renderTrain({json.dumps({"curves":img_curves,"summary":summary})})')
    log("✅ Dashboard actualizado.", "ok")
    prog(100)

# ══════════════════════════════════════════════════════
#  10 — DASHBOARD HTML
# ══════════════════════════════════════════════════════
_CHIPS_HTML = "\n".join(
    f'<button class="chip{"  chip-on" if c in CLASES_DEF else ""}" '
    f'data-cls="{c}">{c}</button>'
    for c in CLASES_COCO
)

_DASHBOARD = r"""<!DOCTYPE html>
<html lang="es">
<head><meta charset="UTF-8">
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@300;400;600&family=Barlow:wght@400;600;700;800;900&display=swap');
*{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#06080c;--card:#0b1018;--card2:#101620;--card3:#16202e;
  --c1:#00e5ff;--c2:#00ff9d;--c3:#ff4466;--c4:#bf7fff;--c5:#ffbe0b;
  --text:#dce8f5;--muted:#4a6070;--border:#18232f;
  --font:'Barlow',sans-serif;--mono:'IBM Plex Mono',monospace;
}
html,body{min-height:100%;background:var(--bg);color:var(--text);
          font-family:var(--font);overflow-x:hidden}

/* TOP */
.top{position:sticky;top:0;z-index:100;display:flex;align-items:center;
  gap:14px;padding:0 22px;height:52px;
  background:rgba(6,8,12,.98);backdrop-filter:blur(20px);
  border-bottom:1px solid var(--border)}
.top::after{content:'';position:absolute;bottom:0;left:0;right:0;height:1px;
  background:linear-gradient(90deg,transparent,var(--c1) 30%,var(--c4) 70%,transparent);
  opacity:.4}
.logo{width:32px;height:32px;border-radius:8px;font-size:16px;flex-shrink:0;
  display:grid;place-items:center;
  background:linear-gradient(135deg,var(--c1),#0055cc);
  box-shadow:0 0 20px rgba(0,229,255,.5)}
.appname{font-size:15px;font-weight:800;letter-spacing:-.3px}
.appname em{color:var(--c1);font-style:normal}
.vtag{font-family:var(--mono);font-size:9px;color:var(--muted);
  background:var(--card2);border:1px solid var(--border);
  border-radius:3px;padding:2px 8px;margin-left:6px}
.topright{margin-left:auto;display:flex;align-items:center;gap:12px}
.sdot{width:7px;height:7px;border-radius:50%;background:var(--muted);flex-shrink:0}
.sdot.on{background:var(--c2);box-shadow:0 0 10px var(--c2);animation:blink 1.3s infinite}
.sdot.warn{background:var(--c5);box-shadow:0 0 8px var(--c5)}
.sdot.err{background:var(--c3);box-shadow:0 0 8px var(--c3)}
.stxt{font-family:var(--mono);font-size:9px;color:var(--muted);letter-spacing:.5px}
@keyframes blink{0%,100%{opacity:1}50%{opacity:.2}}

/* LAYOUT */
.layout{display:grid;grid-template-columns:268px 1fr;
  min-height:calc(100vh - 52px)}

/* SIDEBAR */
.sidebar{background:var(--card);border-right:1px solid var(--border);
  display:flex;flex-direction:column;
  position:sticky;top:52px;height:calc(100vh - 52px);overflow:hidden}
.sb-scroll{flex:1;overflow-y:auto;padding:15px 13px;
  display:flex;flex-direction:column;gap:13px}
.sb-scroll::-webkit-scrollbar{width:3px}
.sb-scroll::-webkit-scrollbar-thumb{background:var(--border)}

.sec{font-family:var(--mono);font-size:8px;letter-spacing:2.5px;
  text-transform:uppercase;color:var(--muted);
  display:flex;align-items:center;gap:8px}
.sec::after{content:'';flex:1;height:1px;background:var(--border)}

select,input[type=text]{
  background:var(--card2);border:1px solid var(--border);
  color:var(--text);border-radius:7px;padding:7px 10px;
  font-size:11px;width:100%;font-family:var(--font);
  transition:border-color .2s;outline:none}
select:focus,input:focus{border-color:var(--c1)}
.frow{display:flex;flex-direction:column;gap:4px}
.frow label{font-size:10px;color:var(--muted);font-family:var(--mono)}
.rh{display:flex;justify-content:space-between;align-items:center;margin-bottom:4px}
.rh label{font-size:10px;color:var(--muted);font-family:var(--mono)}
.rv{font-family:var(--mono);font-size:11px;color:var(--c1);
  background:rgba(0,229,255,.07);border:1px solid rgba(0,229,255,.15);
  border-radius:4px;padding:1px 7px;min-width:40px;text-align:center}
input[type=range]{width:100%;accent-color:var(--c1);cursor:pointer}

.chips{display:flex;flex-wrap:wrap;gap:4px;max-height:158px;overflow-y:auto;padding:1px}
.chips::-webkit-scrollbar{width:3px}
.chips::-webkit-scrollbar-thumb{background:var(--border)}
.chip{font-family:var(--mono);font-size:9px;padding:2px 8px;
  border-radius:10px;border:1px solid var(--border);
  background:var(--card2);color:var(--muted);cursor:pointer;
  transition:all .12s;user-select:none}
.chip-on{background:rgba(0,229,255,.09);border-color:var(--c1);color:var(--c1)}
.chip:hover{border-color:var(--muted);color:var(--text)}
.divr{height:1px;background:var(--border);margin:2px 0}

/* SIDEBAR FOOT */
.sbfoot{padding:12px 13px;border-top:1px solid var(--border);
  display:flex;flex-direction:column;gap:6px}
.btn{width:100%;padding:10px 14px;border-radius:8px;border:none;cursor:pointer;
  font-size:12px;font-weight:700;font-family:var(--font);
  display:flex;align-items:center;justify-content:center;gap:7px;
  transition:all .18s;letter-spacing:.2px}
.btn-cam{background:var(--card2);border:1px solid var(--border);color:var(--text)}
.btn-cam:hover{border-color:var(--c2);color:var(--c2)}
.btn-cam.on{background:rgba(0,255,157,.08);border-color:var(--c2);color:var(--c2)}
.btn-run{background:linear-gradient(135deg,var(--c1) 0%,#0060d0 100%);
  color:#000;font-size:13px;font-weight:800;
  box-shadow:0 4px 22px rgba(0,229,255,.3);letter-spacing:.3px}
.btn-run:hover:not(:disabled){transform:translateY(-2px);
  box-shadow:0 12px 36px rgba(0,229,255,.5)}
.btn-run:disabled{opacity:.28;cursor:not-allowed;transform:none;box-shadow:none}
.btn-train{background:linear-gradient(135deg,var(--c4) 0%,#7030cc 100%);
  color:#fff;font-size:12px;font-weight:700;
  box-shadow:0 4px 18px rgba(191,127,255,.3)}
.btn-train:hover:not(:disabled){transform:translateY(-1px);
  box-shadow:0 10px 28px rgba(191,127,255,.45)}
.btn-train:disabled{opacity:.28;cursor:not-allowed;transform:none;box-shadow:none}
.pbar-bg{height:3px;background:var(--border);border-radius:2px;overflow:hidden}
.pbar{height:100%;width:0%;border-radius:2px;
  background:linear-gradient(90deg,var(--c1),var(--c2));
  transition:width .4s;box-shadow:0 0 8px rgba(0,229,255,.6)}

/* MAIN */
.main{display:flex;flex-direction:column;overflow:hidden}

/* CAMERA ROW */
.camrow{display:grid;grid-template-columns:1fr 1fr;
  height:40vh;border-bottom:1px solid var(--border);flex-shrink:0}
.camp,.detp{position:relative;overflow:hidden;background:#000}
.camp{border-right:1px solid var(--border)}
.plbl{position:absolute;top:9px;left:11px;z-index:10;
  font-family:var(--mono);font-size:9px;letter-spacing:.8px;
  padding:3px 9px;border-radius:4px;border:1px solid;
  background:rgba(6,8,12,.9);backdrop-filter:blur(4px)}
.plbl-c{color:var(--c2);border-color:rgba(0,255,157,.28)}
.plbl-d{color:var(--c1);border-color:rgba(0,229,255,.25)}
video,canvas{width:100%;height:100%;object-fit:cover;display:block}
.fps-p{position:absolute;bottom:9px;right:11px;z-index:10;
  font-family:var(--mono);font-size:9px;color:var(--c2);
  background:rgba(6,8,12,.9);border:1px solid rgba(0,255,157,.2);
  border-radius:4px;padding:2px 8px}
.det-cnt{position:absolute;bottom:9px;left:11px;z-index:10;
  font-family:var(--mono);font-size:9px;color:var(--c1);
  background:rgba(6,8,12,.9);border:1px solid rgba(0,229,255,.2);
  border-radius:4px;padding:2px 8px}
.ov{position:absolute;inset:0;display:flex;flex-direction:column;
  align-items:center;justify-content:center;gap:8px;color:var(--muted);
  pointer-events:none}
.ov-i{font-size:28px;opacity:.2}
.ov-t{font-size:10px;opacity:.35;font-family:var(--mono)}

/* TABS */
.tabs{display:flex;border-bottom:1px solid var(--border);
  background:var(--card);flex-shrink:0}
.tab{font-family:var(--mono);font-size:10px;letter-spacing:1px;
  text-transform:uppercase;padding:11px 20px;cursor:pointer;
  color:var(--muted);border-bottom:2px solid transparent;
  transition:all .15s;user-select:none}
.tab:hover{color:var(--text)}
.tab.on{color:var(--c1);border-bottom-color:var(--c1)}

/* PANELS */
.panel{display:none;padding:15px 19px;flex-direction:column;gap:11px;
  overflow-y:auto;flex:1}
.panel.on{display:flex}
.panel::-webkit-scrollbar{width:3px}
.panel::-webkit-scrollbar-thumb{background:var(--border)}

/* KPI */
.krow{display:grid;grid-template-columns:repeat(4,1fr);gap:8px}
.kpi{background:var(--card);border:1px solid var(--border);border-radius:10px;
  padding:13px 15px;position:relative;overflow:hidden;transition:transform .18s}
.kpi:hover{transform:translateY(-2px)}
.kpi::before{content:'';position:absolute;top:0;left:0;right:0;height:2px}
.k1::before{background:var(--c1)} .k2::before{background:var(--c2)}
.k3::before{background:var(--c3)} .k4::before{background:var(--c4)}
.kpi-l{font-family:var(--mono);font-size:8px;letter-spacing:1.5px;
  text-transform:uppercase;color:var(--muted);margin-bottom:7px}
.kpi-v{font-size:26px;font-weight:800;line-height:1;letter-spacing:-1.5px}
.k1 .kpi-v{color:var(--c1)} .k2 .kpi-v{color:var(--c2)}
.k3 .kpi-v{color:var(--c3)} .k4 .kpi-v{color:var(--c4)}
.kpi-s{font-family:var(--mono);font-size:9px;color:var(--muted);margin-top:4px}

/* STAT BARS */
.sbars{display:flex;flex-direction:column;gap:8px}
.sbar{display:flex;align-items:center;gap:10px}
.sbn{font-family:var(--mono);font-size:9px;color:var(--muted);width:76px;flex-shrink:0}
.sbb{flex:1;height:5px;background:var(--card2);border-radius:3px;overflow:hidden}
.sbf{height:100%;border-radius:3px;transition:width .8s ease}
.sbv{font-family:var(--mono);font-size:10px;color:var(--text);width:40px;text-align:right}

/* CHART BOXES */
.ch2{display:grid;grid-template-columns:2fr 1fr;gap:8px}
.ch3{display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px}
.cbox{background:var(--card);border:1px solid var(--border);
  border-radius:10px;padding:13px 15px}
.cbox-t{font-family:var(--mono);font-size:8px;letter-spacing:2px;
  text-transform:uppercase;color:var(--muted);margin-bottom:10px}
.cbox img{width:100%;border-radius:5px;display:block}

/* REPORT */
.rpt{background:var(--card2);border:1px solid var(--border);
  border-left:2px solid var(--c1);border-radius:7px;
  padding:12px 14px;font-family:var(--mono);font-size:9.5px;
  color:var(--text);white-space:pre;overflow-x:auto;line-height:1.8}
.demo-w{background:rgba(255,68,102,.07);border:1px solid rgba(255,68,102,.22);
  border-radius:7px;padding:9px 13px;font-family:var(--mono);font-size:9px;
  color:var(--c3);display:flex;align-items:center;gap:8px}

/* TRAIN INFO */
.tinfo{background:var(--card2);border:1px solid var(--border);
  border-left:2px solid var(--c4);border-radius:7px;
  padding:13px 15px;font-family:var(--mono);font-size:10px;
  color:var(--text);white-space:pre-wrap;line-height:1.8}

/* CONSOLE */
.conbar{border-top:1px solid var(--border);padding:7px 19px;
  display:flex;flex-direction:column;gap:3px;flex-shrink:0}
.con-l{font-family:var(--mono);font-size:8px;letter-spacing:2px;
  text-transform:uppercase;color:var(--muted)}
.con{height:52px;overflow-y:auto;font-family:var(--mono);font-size:10px;
  color:var(--muted);line-height:1.75}
.con::-webkit-scrollbar{width:3px}
.con::-webkit-scrollbar-thumb{background:var(--border)}
.log-ok{color:var(--c2)} .log-err{color:var(--c3)}
.log-info{color:var(--c1)} .log-warn{color:var(--c5)}

/* EMPTY */
.empty{display:flex;flex-direction:column;align-items:center;
  justify-content:center;min-height:130px;gap:8px}
.ei{font-size:32px;opacity:.18}
.et{font-size:11px;color:var(--muted);opacity:.4;font-family:var(--mono)}

/* FILE */
#file-wrap{display:none;margin-top:6px}

/* SPINNER */
@keyframes spin{to{transform:rotate(360deg)}}
.spin{width:12px;height:12px;border:2px solid rgba(0,0,0,.25);
  border-top-color:#000;border-radius:50%;
  animation:spin .6s linear infinite;display:inline-block}
.spin-w{width:12px;height:12px;border:2px solid rgba(255,255,255,.2);
  border-top-color:#fff;border-radius:50%;
  animation:spin .6s linear infinite;display:inline-block}

@keyframes fu{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:none}}
.fu{animation:fu .35s ease both}
</style>
</head>
<body>

<!-- TOPBAR -->
<div class="top">
  <div class="logo">👁</div>
  <div><span class="appname">YOLOv8 <em>Vision Studio</em></span>
    <span class="vtag">v9.0</span></div>
  <div class="topright">
    <div class="sdot" id="sdot"></div>
    <div class="stxt" id="stxt">IDLE</div>
  </div>
</div>

<div class="layout">

<!-- SIDEBAR -->
<aside class="sidebar">
  <div class="sb-scroll">

    <div>
      <div class="sec">Modelo</div>
      <div class="frow"><label>Variante YOLOv8</label>
        <select id="sel-model">
          <option value="yolov8n" selected>yolov8n — Nano ⚡ más rápido</option>
          <option value="yolov8s">yolov8s — Small ⭐</option>
          <option value="yolov8m">yolov8m — Medium 🎯</option>
        </select>
      </div>
      <div style="margin-top:10px">
        <div class="rh"><label>Confianza mínima</label>
          <span class="rv" id="rv-conf">0.25</span></div>
        <input type="range" id="sl-conf" min="0.05" max="0.90" step="0.05" value="0.25"
          oninput="document.getElementById('rv-conf').textContent=parseFloat(this.value).toFixed(2)">
      </div>
    </div>

    <div class="divr"></div>

    <div>
      <div class="sec">Captura</div>
      <div class="frow"><label>Fuente</label>
        <select id="sel-src" onchange="onSrc()">
          <option value="camera">📷 Cámara web</option>
          <option value="demo">🎬 Video demo</option>
          <option value="file">📁 Archivo</option>
        </select>
      </div>
      <div id="file-wrap">
        <div class="frow" style="margin-top:6px"><label>Ruta</label>
          <input type="text" id="inp-file" placeholder="/content/video.mp4">
        </div>
      </div>
      <div style="margin-top:10px">
        <div class="rh"><label>Frames a analizar</label>
          <span class="rv" id="rv-frames">25</span></div>
        <input type="range" id="sl-frames" min="5" max="80" step="5" value="25"
          oninput="document.getElementById('rv-frames').textContent=this.value">
      </div>
    </div>

    <div class="divr"></div>

    <div>
      <div class="sec">Clases de interés</div>
      <div class="chips" id="chips">__CHIPS__</div>
    </div>

    <div class="divr"></div>

    <div>
      <div class="sec">Entrenamiento</div>
      <div class="frow"><label>Épocas</label>
        <select id="sel-epochs">
          <option value="5">5 — test rápido</option>
          <option value="10">10 épocas</option>
          <option value="20" selected>20 épocas — recomendado</option>
          <option value="50">50 épocas</option>
          <option value="100">100 épocas</option>
        </select>
      </div>
      <div class="frow" style="margin-top:6px"><label>Dataset YAML</label>
        <input type="text" id="inp-yaml" value="coco128.yaml" placeholder="dataset.yaml">
      </div>
    </div>

  </div>

  <div class="sbfoot">
    <button class="btn btn-cam" id="btn-cam" onclick="toggleCam()">
      📷 &nbsp;INICIAR CÁMARA
    </button>
    <button class="btn btn-run" id="btn-run" onclick="doAnalizar()" disabled>
      ⚡ &nbsp;ANALIZAR
    </button>
    <button class="btn btn-train" id="btn-train" onclick="doEntrenar()">
      🧠 &nbsp;ENTRENAR
    </button>
    <div class="pbar-bg"><div class="pbar" id="pbar"></div></div>
  </div>
</aside>

<!-- MAIN -->
<div class="main">

  <!-- CAMERA ROW -->
  <div class="camrow">
    <div class="camp">
      <div class="plbl plbl-c">● CÁMARA EN VIVO</div>
      <div class="fps-p" id="fps-p" style="display:none">0 fps</div>
      <video id="cam-vid" autoplay playsinline muted></video>
      <div class="ov" id="cam-ov">
        <div class="ov-i">📷</div>
        <div class="ov-t">Pulsa "INICIAR CÁMARA"</div>
      </div>
    </div>
    <div class="detp">
      <div class="plbl plbl-d">⬡ DETECCIÓN YOLOv8</div>
      <div class="det-cnt" id="det-cnt" style="display:none">— objetos</div>
      <canvas id="det-canvas"></canvas>
      <div class="ov" id="det-ov">
        <div class="ov-i">🔍</div>
        <div class="ov-t">Detecciones aparecen aquí</div>
      </div>
    </div>
  </div>

  <!-- TABS -->
  <div class="tabs">
    <div class="tab on" id="tab-det" onclick="switchTab('det')">📊 &nbsp;Detección</div>
    <div class="tab"    id="tab-train" onclick="switchTab('train')">🧠 &nbsp;Entrenamiento</div>
  </div>

  <!-- PANEL DETECCIÓN -->
  <div class="panel on" id="panel-det">
    <div class="empty">
      <div class="ei">📊</div>
      <div class="et">Inicia la cámara → pulsa ⚡ ANALIZAR</div>
    </div>
  </div>

  <!-- PANEL ENTRENAMIENTO -->
  <div class="panel" id="panel-train">
    <div class="tinfo" id="tinfo">Pulsa 🧠 ENTRENAR para iniciar el entrenamiento.

El sistema usará el dataset indicado en "Dataset YAML".
Para test rápido usa "coco128.yaml" (ya incluido en ultralytics).

Para tu propio dataset:
  1. Sube las imágenes a /content/mi_dataset/
  2. Crea un archivo .yaml con rutas y clases
  3. Escribe la ruta en el campo "Dataset YAML"
  4. Pulsa ENTRENAR

El progreso aparecerá aquí y en la consola de Colab.</div>
  </div>

  <!-- CONSOLE -->
  <div class="conbar">
    <div class="con-l">● Consola</div>
    <div class="con" id="con">
      <span class="log-info">[sistema] YOLOv8 Vision Studio v9.0 listo.</span>
    </div>
  </div>

</div><!-- /main -->
</div><!-- /layout -->

<script>
// ── GLOBALS ──────────────────────────────────────────────
let stream=null, camActive=false, fpsTimer=null, lastT=0, fpsVal=0;

// ── UI NAMESPACE (llamado por Python via eval_js) ─────────
const UI = {
  log(msg, cls=''){
    const c=document.getElementById('con');
    const s=document.createElement('span');
    s.className='log-'+cls; s.textContent=`[${ts()}] ${msg}\n`;
    c.appendChild(s); c.scrollTop=c.scrollHeight;
  },
  prog(v){ document.getElementById('pbar').style.width=v+'%'; },
  status(t, live=true){
    document.getElementById('stxt').textContent=t;
    const d=document.getElementById('sdot');
    d.className='sdot'+(live?' on':'');
  },
  frame(b64, count){
    document.getElementById('det-ov').style.display='none';
    const cv=document.getElementById('det-canvas');
    const img=new Image();
    img.onload=()=>{cv.width=img.width;cv.height=img.height;
                     cv.getContext('2d').drawImage(img,0,0)};
    img.src='data:image/jpeg;base64,'+b64;
    const p=document.getElementById('det-cnt');
    p.style.display='block';
    p.textContent=count+' objeto'+(count!==1?'s':'');
  },
  trainInfo(txt){ document.getElementById('tinfo').textContent=txt; },

  renderDet(d){
    const m=d.metrics;
    const mk=(n,v,c)=>`
      <div class="sbar">
        <span class="sbn">${n}</span>
        <div class="sbb"><div class="sbf" style="width:${Math.round(v*100)}%;background:${c}"></div></div>
        <span class="sbv">${Math.round(v*100)}%</span>
      </div>`;
    const demoW=d.demo?'<div class="demo-w fu">⚠ Sin detecciones reales — mostrando datos de demostración sintéticos</div>':'';
    const html=`
      <div class="krow fu">
        <div class="kpi k1"><div class="kpi-l">Accuracy</div>
          <div class="kpi-v">${(m.accuracy*100).toFixed(1)}%</div>
          <div class="kpi-s">${m.accuracy.toFixed(6)}</div></div>
        <div class="kpi k2"><div class="kpi-l">Precision</div>
          <div class="kpi-v">${(m.precision*100).toFixed(1)}%</div>
          <div class="kpi-s">macro avg</div></div>
        <div class="kpi k3"><div class="kpi-l">Recall</div>
          <div class="kpi-v">${(m.recall*100).toFixed(1)}%</div>
          <div class="kpi-s">macro avg</div></div>
        <div class="kpi k4"><div class="kpi-l">F1-Score</div>
          <div class="kpi-v">${(m.f1*100).toFixed(1)}%</div>
          <div class="kpi-s">harmonic mean</div></div>
      </div>
      <div class="cbox fu"><div class="cbox-t">Resumen de métricas</div>
        <div class="sbars">
          ${mk('Accuracy', m.accuracy,'#00e5ff')}
          ${mk('Precision',m.precision,'#00ff9d')}
          ${mk('Recall',   m.recall,   '#ff4466')}
          ${mk('F1-Score', m.f1,       '#bf7fff')}
        </div>
      </div>
      ${demoW}
      <div class="ch2 fu">
        <div class="cbox"><div class="cbox-t">Barras de métricas</div>
          <img src="data:image/png;base64,${d.bars}"></div>
        <div class="cbox"><div class="cbox-t">Radar</div>
          <img src="data:image/png;base64,${d.radar}"></div>
      </div>
      <div class="ch3 fu">
        <div class="cbox"><div class="cbox-t">Matriz de confusión</div>
          <img src="data:image/png;base64,${d.cm}"></div>
        <div class="cbox"><div class="cbox-t">Detecciones / frame</div>
          <img src="data:image/png;base64,${d.timeline}"></div>
        <div class="cbox"><div class="cbox-t">Distribución clases</div>
          <img src="data:image/png;base64,${d.pie}"></div>
      </div>
      ${d.box?`<div class="cbox fu"><div class="cbox-t">Confianza por clase</div>
        <img src="data:image/png;base64,${d.box}"></div>`:''}
      <div class="cbox fu"><div class="cbox-t">Reporte sklearn</div>
        <pre class="rpt">${d.report}</pre></div>`;
    document.getElementById('panel-det').innerHTML=html;
    document.getElementById('btn-run').disabled=false;
    document.getElementById('btn-run').innerHTML='⚡ &nbsp;ANALIZAR';
    UI.status('ANÁLISIS COMPLETO ✓',false);
    UI.prog(100); setTimeout(()=>UI.prog(0),3000);
  },

  renderTrain(d){
    const html=`
      ${d.curves?`<div class="cbox fu"><div class="cbox-t">Curvas de entrenamiento</div>
        <img src="data:image/png;base64,${d.curves}"></div>`:''}
      <div class="cbox fu"><div class="cbox-t">Resumen</div>
        <pre class="rpt">${d.summary}</pre></div>`;
    document.getElementById('panel-train').innerHTML=html;
    document.getElementById('btn-train').disabled=false;
    document.getElementById('btn-train').innerHTML='🧠 &nbsp;ENTRENAR';
    UI.status('ENTRENAMIENTO COMPLETO ✓',false);
    UI.prog(100); setTimeout(()=>UI.prog(0),3000);
  }
};

function ts(){return new Date().toLocaleTimeString('es',{hour12:false})}

// ── TABS ──────────────────────────────────────────────────
function switchTab(id){
  ['det','train'].forEach(t=>{
    document.getElementById('tab-'+t).classList.toggle('on',t===id);
    document.getElementById('panel-'+t).classList.toggle('on',t===id);
  });
}

// ── CHIPS ─────────────────────────────────────────────────
document.querySelectorAll('.chip').forEach(c=>{c.onclick=()=>c.classList.toggle('chip-on')});
function getClases(){return [...document.querySelectorAll('.chip.chip-on')].map(c=>c.dataset.cls)}

// ── SOURCE ────────────────────────────────────────────────
function onSrc(){
  const v=document.getElementById('sel-src').value;
  document.getElementById('file-wrap').style.display=v==='file'?'block':'none';
  if(v!=='camera'){stopCam();document.getElementById('btn-run').disabled=false;}
  else document.getElementById('btn-run').disabled=!camActive;
}

// ── CAMERA ────────────────────────────────────────────────
async function toggleCam(){camActive?stopCam():await startCam()}
async function startCam(){
  try{
    stream=await navigator.mediaDevices.getUserMedia({
      video:{width:{ideal:640},height:{ideal:480},frameRate:{ideal:30}},audio:false});
    const v=document.getElementById('cam-vid');
    v.srcObject=stream; await v.play();
    document.getElementById('cam-ov').style.display='none';
    document.getElementById('fps-p').style.display='block';
    document.getElementById('btn-cam').innerHTML='⏹ &nbsp;DETENER CÁMARA';
    document.getElementById('btn-cam').classList.add('on');
    document.getElementById('btn-run').disabled=false;
    camActive=true; UI.status('CÁMARA ACTIVA',true);
    UI.log('Cámara iniciada.','ok');
    fpsTimer=setInterval(()=>{
      document.getElementById('fps-p').textContent=fpsVal+' fps';
    },700);
  }catch(e){
    UI.log('Cámara denegada: '+e.message,'err');
    UI.log('Permite el acceso en la barra de URL del navegador.','warn');
    UI.status('SIN CÁMARA',false);
  }
}
function stopCam(){
  if(stream)stream.getTracks().forEach(t=>t.stop());
  stream=null;camActive=false;
  if(fpsTimer)clearInterval(fpsTimer);
  document.getElementById('cam-vid').srcObject=null;
  document.getElementById('cam-ov').style.display='flex';
  document.getElementById('fps-p').style.display='none';
  document.getElementById('btn-cam').innerHTML='📷 &nbsp;INICIAR CÁMARA';
  document.getElementById('btn-cam').classList.remove('on');
  document.getElementById('btn-run').disabled=true;
  UI.status('IDLE',false); UI.log('Cámara detenida.','info');
}
function snapFrame(){
  const v=document.getElementById('cam-vid');
  if(!v.srcObject)return null;
  const c=document.createElement('canvas');
  c.width=v.videoWidth||640;c.height=v.videoHeight||480;
  c.getContext('2d').drawImage(v,0,0,c.width,c.height);
  const now=performance.now();fpsVal=Math.round(1000/(now-lastT+1));lastT=now;
  return c.toDataURL('image/jpeg',0.82);
}

// ── ANALIZAR ─────────────────────────────────────────────
async function doAnalizar(){
  const clases=getClases();
  if(!clases.length){UI.log('Selecciona al menos una clase.','warn');return;}
  const src=document.getElementById('sel-src').value;
  if(src==='camera'&&!camActive){UI.log('Inicia la cámara primero.','warn');return;}

  const params=JSON.stringify({
    src,
    modelo:  document.getElementById('sel-model').value,
    conf:    parseFloat(document.getElementById('sl-conf').value),
    nframes: parseInt(document.getElementById('sl-frames').value),
    clases,
    fpath:   document.getElementById('inp-file').value,
  });

  const btn=document.getElementById('btn-run');
  btn.disabled=true; btn.innerHTML='<span class="spin"></span> &nbsp;Analizando...';
  UI.status('ANALIZANDO',true); UI.prog(2);
  UI.log(`Enviando al kernel Python: ${src} | ${document.getElementById('sel-model').value}`,'info');
  switchTab('det');

  try{
    await google.colab.kernel.invokeFunction('vs9_analizar',[params],{});
  }catch(e){
    UI.log('Error kernel: '+e.message+' — ejecuta analizar_manual() en nueva celda','err');
    btn.disabled=false; btn.innerHTML='⚡ &nbsp;ANALIZAR';
    UI.status('ERROR',false);
    // Guardar config por si el usuario ejecuta manualmente
    window.__cfg_manual__=params;
  }
}

// ── ENTRENAR ─────────────────────────────────────────────
async function doEntrenar(){
  const params=JSON.stringify({
    epochs:  parseInt(document.getElementById('sel-epochs').value),
    yaml:    document.getElementById('inp-yaml').value.trim()||'coco128.yaml',
    modelo:  document.getElementById('sel-model').value,
    imgsize: 640, batch: 16,
  });

  const btn=document.getElementById('btn-train');
  btn.disabled=true; btn.innerHTML='<span class="spin-w"></span> &nbsp;Entrenando...';
  UI.status('ENTRENANDO',true); UI.prog(2);
  switchTab('train');
  document.getElementById('tinfo').textContent=
    'Iniciando entrenamiento...\nRevisa la consola de Colab para el progreso.';

  try{
    await google.colab.kernel.invokeFunction('vs9_entrenar',[params],{});
  }catch(e){
    UI.log('Error kernel: '+e.message+' — ejecuta entrenar_manual() en nueva celda','err');
    btn.disabled=false; btn.innerHTML='🧠 &nbsp;ENTRENAR';
    UI.status('ERROR',false);
    window.__train_manual__=params;
  }
}
</script>
</body></html>
"""

# ══════════════════════════════════════════════════════
#  11 — LANZAR DASHBOARD + REGISTRAR CALLBACKS
# ══════════════════════════════════════════════════════
def _launch():
    from google.colab import output as cout

    # Registrar los callbacks que JS invoca al pulsar los botones
    cout.register_callback("vs9_analizar",  _cb_analizar)
    cout.register_callback("vs9_entrenar",  _cb_entrenar)
    print("🔗  Callbacks registrados: vs9_analizar, vs9_entrenar")

    html = _DASHBOARD.replace("__CHIPS__", _CHIPS_HTML)
    display(HTML(html))

    print()
    print("╔" + "═"*60 + "╗")
    print("║   YOLOv8 Vision Studio v9.0  ─  Listo                     ║")
    print("╠" + "═"*60 + "╣")
    print("║                                                            ║")
    print("║  1. Pulsa 📷 INICIAR CÁMARA  →  acepta permiso             ║")
    print("║  2. Pulsa ⚡ ANALIZAR         →  resultado automático       ║")
    print("║  3. Pulsa 🧠 ENTRENAR         →  fine-tuning automático     ║")
    print("║                                                            ║")
    print("║  Si el botón da error, ejecuta en nueva celda:            ║")
    print("║     analizar_manual()   ó   entrenar_manual()             ║")
    print("║                                                            ║")
    print("╚" + "═"*60 + "╝")


def analizar_manual(fuente="demo", modelo="yolov8n", conf=0.25,
                    clases=None, n_frames=25):
    """Fallback manual — ejecuta en nueva celda si el botón no responde."""
    p = json.dumps({
        "src": fuente, "modelo": modelo, "conf": conf,
        "nframes": n_frames,
        "clases": clases or CLASES_DEF,
        "fpath": "",
    })
    _cb_analizar(p)


def entrenar_manual(yaml="coco128.yaml", modelo="yolov8n",
                    epocas=20, batch=16):
    """Fallback manual — ejecuta en nueva celda si el botón no responde."""
    p = json.dumps({
        "yaml": yaml, "modelo": modelo,
        "epochs": epocas, "imgsize": 640, "batch": batch,
    })
    _cb_entrenar(p)


# ══════════════════════════════════════════════════════
#  12 — EJECUTAR
# ══════════════════════════════════════════════════════
_launch()

⚙️  Instalando dependencias...
✅  Listo.

🧠  Pre-descargando yolov8n (detección rápida)...
✅  Modelo listo.

🔗  Callbacks registrados: vs9_analizar, vs9_entrenar



╔════════════════════════════════════════════════════════════╗
║   YOLOv8 Vision Studio v9.0  ─  Listo                     ║
╠════════════════════════════════════════════════════════════╣
║                                                            ║
║  1. Pulsa 📷 INICIAR CÁMARA  →  acepta permiso             ║
║  2. Pulsa ⚡ ANALIZAR         →  resultado automático       ║
║  3. Pulsa 🧠 ENTRENAR         →  fine-tuning automático     ║
║                                                            ║
║  Si el botón da error, ejecuta en nueva celda:            ║
║     analizar_manual()   ó   entrenar_manual()             ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝

════════════════════════════════════════════════════
  ANÁLISIS  |  yolov8n  |  conf=0.25  |  src=camera
════════════════════════════════════════════════════
  🔹  Cargando yolov8n...
  ✅  Modelo 'yolov8n' listo.
  📷  Capturando 25 frames de cámar